# WiDS Wildfire Survival — Rank-Gated Horizon Ensemble

In [1]:

import os, glob, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAS_CAT = True
except Exception:
    HAS_CAT = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

SEED = 20260423
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")
REQUIRED = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
HORIZONS = [12, 24, 48]
SEEDS = [11, 37, 79, 123]


def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wids-global-datathon-2026",
        "/kaggle/input",
        ".",
        "/mnt/data",
    ]
    needed = {"train.csv", "test.csv", "sample_submission.csv"}
    for root in candidates:
        if not os.path.isdir(root):
            continue
        if needed.issubset(set(os.listdir(root))):
            return root
    for root, _, files in os.walk("/kaggle/input" if os.path.isdir("/kaggle/input") else "."):
        if needed.issubset(set(files)):
            return root
    raise FileNotFoundError("Cannot find train.csv/test.csv/sample_submission.csv")

DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(x, dtype=float), -40, 40)))


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1.0 - 1e-7)
    return np.log(p / (1.0 - p))


def compute_c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    conc = 0.0
    comp = 0.0
    for i in range(len(time)):
        if event[i] != 1:
            continue
        m = time[i] < time
        if not np.any(m):
            continue
        comp += float(np.sum(m))
        conc += float(np.sum(risk[i] > risk[m])) + 0.5 * float(np.sum(risk[i] == risk[m]))
    return conc / comp if comp > 0 else 0.5


def compute_brier(time, event, prob, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    prob = np.clip(np.asarray(prob, dtype=float), 0.0, 1.0)
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)
    if valid.sum() == 0:
        return 0.25
    return float(np.mean((prob[valid] - y[valid]) ** 2))


def hybrid_score(time, event, P):
    risk = 0.3 * P[:, 1] + 0.4 * P[:, 2] + 0.3 * P[:, 3]
    cidx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, P[:, 1], 24)
    b48 = compute_brier(time, event, P[:, 2], 48)
    b72 = compute_brier(time, event, P[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * cidx + 0.7 * (1.0 - wb), cidx, wb, b24, b48, b72


def enforce_monotone(P):
    P = np.clip(np.asarray(P, dtype=float), 0.0, 1.0)
    for j in range(1, P.shape[1]):
        P[:, j] = np.maximum(P[:, j], P[:, j - 1])
    return P


def add_features(df):
    x = df.copy()
    for c in x.columns:
        if c != "event_id":
            x[c] = pd.to_numeric(x[c], errors="coerce")

    dist = x["dist_min_ci_0_5h"].clip(lower=1.0)
    km = dist / 1000.0
    area = x["area_first_ha"].clip(lower=0.0)
    growth_abs = x["area_growth_abs_0_5h"].clip(lower=0.0)
    growth_rate = x["area_growth_rate_ha_per_h"]
    per = x["num_perimeters_0_5h"]
    dt = x["dt_first_last_0_5h"]
    low = x["low_temporal_resolution_0_5h"]
    close = x["closing_speed_m_per_h"]
    close_pos = close.clip(lower=0.0)
    close_neg = (-close).clip(lower=0.0)
    radial = x["radial_growth_rate_m_per_h"].clip(lower=0.0)
    along = x["along_track_speed"].clip(lower=0.0)
    advance = x["projected_advance_m"].clip(lower=0.0)
    align = x["alignment_abs"].clip(0.0, 1.0)
    radius = np.sqrt(area * 10000.0 / np.pi)
    eff_close = close_pos + 0.35 * radial * align + 0.25 * along + 1.0

    x["dist_km"] = km
    x["log_dist"] = np.log1p(dist)
    x["sqrt_dist"] = np.sqrt(dist)
    x["inv_dist"] = 1.0 / (km + 0.1)
    x["inv_dist_sq"] = x["inv_dist"] ** 2
    x["gap_to_5km"] = (dist - 5000.0).clip(lower=0.0)
    x["under_5km"] = (5000.0 - dist).clip(lower=0.0)
    x["log_under_5km"] = np.log1p(x["under_5km"])
    x["near_5km_flag"] = (dist < 5000.0).astype(float)
    x["near_3km_flag"] = (dist < 3000.0).astype(float)
    x["near_2km_flag"] = (dist < 2000.0).astype(float)

    x["log_area"] = np.log1p(area)
    x["fire_radius_m"] = radius
    x["fire_radius_km"] = radius / 1000.0
    x["radius_to_dist"] = radius / (dist + 1.0)
    x["area_to_dist"] = area / (km + 0.1)
    x["log_area_dist"] = np.log1p(area) - np.log1p(dist)
    x["growth_to_area"] = growth_abs / (area + 1.0)
    x["growth_to_dist"] = growth_rate.clip(lower=0.0) / (km + 0.2)
    x["radial_to_dist"] = radial / (dist + 100.0)

    x["closing_pos"] = close_pos
    x["closing_neg"] = close_neg
    x["along_pos"] = along
    x["advance_pos"] = advance
    x["effective_closing_speed"] = eff_close
    x["eta_dist"] = (dist / eff_close).clip(0.0, 9999.0)
    x["log_eta_dist"] = np.log1p(x["eta_dist"])
    x["eta_gap5"] = (x["gap_to_5km"] / eff_close).clip(0.0, 9999.0)
    x["log_eta_gap5"] = np.log1p(x["eta_gap5"])
    x["close_align"] = close_pos * align
    x["radial_align"] = radial * align
    x["advance_align"] = advance * align
    x["multi_align"] = (1.0 - low) * align
    x["quality_close"] = (1.0 - low) * close_pos
    x["low_logarea"] = low * np.log1p(area)

    x["per_log"] = np.log1p(per)
    x["dt_per"] = dt * per
    x["one_perim"] = (per <= 1).astype(float)
    x["multi_perim"] = (per > 1).astype(float)
    x["perim_density"] = per / (dt + 0.25)

    hour = x["event_start_hour"]
    month = x["event_start_month"]
    dow = x["event_start_dayofweek"]
    x["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    x["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    x["month_sin"] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0)
    x["month_cos"] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0)
    x["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0)
    x["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0)
    x["late_hour_flag"] = hour.isin([16, 17, 18, 19]).astype(float)
    x["night_flag"] = hour.isin([0, 1, 2, 3, 4, 5]).astype(float)
    x["hot_response_hour"] = hour.isin([20, 21, 22, 23]).astype(float)
    x["summer_flag"] = month.isin([6, 7, 8]).astype(float)
    x["low_late_hour"] = low * x["late_hour_flag"]
    x["low_night"] = low * x["night_flag"]
    x["low_hot_response_hour"] = low * x["hot_response_hour"]
    x["low_month7"] = low * (month == 7).astype(float)
    x["low_month8"] = low * (month == 8).astype(float)

    for h in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]:
        x[f"hour_{h}"] = (hour == h).astype(float)
    for m in [1, 3, 4, 5, 6, 7, 8, 9]:
        x[f"month_{m}"] = (month == m).astype(float)

    x["hour_bucket"] = pd.cut(hour, bins=[-1, 5, 11, 15, 19, 23], labels=False).astype(float)
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return x

all_features = add_features(pd.concat([train.drop(columns=["event", "time_to_hit_hours"]), test], axis=0, ignore_index=True))
X = all_features.iloc[:len(train)].reset_index(drop=True)
X_test = all_features.iloc[len(train):].reset_index(drop=True)
FEATURES = [c for c in X.columns if c != "event_id"]

TIME = train["time_to_hit_hours"].to_numpy(dtype=float)
EVENT = train["event"].to_numpy(dtype=int)
NEAR = (train["dist_min_ci_0_5h"].to_numpy(dtype=float) < 5000.0)
NEAR_TEST = (test["dist_min_ci_0_5h"].to_numpy(dtype=float) < 5000.0)
NEAR_IDX = np.where(NEAR)[0]
TEST_NEAR_IDX = np.where(NEAR_TEST)[0]

print("DATA_DIR", DATA_DIR)
print("train/test", train.shape, test.shape)
print("train event sum", int(EVENT.sum()), "train <5km", int(NEAR.sum()), "test <5km", int(NEAR_TEST.sum()))


def classifier_bank(seed):
    bank = [
        ("lr_bal", Pipeline([("sc", RobustScaler()), ("lr", LogisticRegression(C=0.25, class_weight="balanced", solver="liblinear", max_iter=600, random_state=seed))])),
        ("lr", Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=0.45, solver="liblinear", max_iter=600, random_state=seed))])),
        ("rf", RandomForestClassifier(n_estimators=120, max_depth=3, min_samples_leaf=3, max_features=0.65, class_weight="balanced_subsample", random_state=seed, n_jobs=1)),
        ("et", ExtraTreesClassifier(n_estimators=180, max_depth=4, min_samples_leaf=2, max_features=0.75, class_weight="balanced", random_state=seed, n_jobs=1)),
        ("gb", GradientBoostingClassifier(n_estimators=100, learning_rate=0.035, max_depth=2, subsample=0.85, random_state=seed)),
        ("ada", AdaBoostClassifier(n_estimators=50, learning_rate=0.05, random_state=seed)),
    ]
    if HAS_LGB:
        bank.append(("lgb", lgb.LGBMClassifier(
            n_estimators=150, learning_rate=0.025, num_leaves=5, max_depth=3,
            min_child_samples=5, subsample=0.85, colsample_bytree=0.80,
            reg_alpha=0.15, reg_lambda=2.0, class_weight="balanced",
            random_state=seed, n_jobs=1, verbose=-1
        )))
    if HAS_CAT:
        bank.append(("cat", CatBoostClassifier(
            iterations=140, learning_rate=0.025, depth=3, l2_leaf_reg=8.0,
            loss_function="Logloss", verbose=False, random_seed=seed,
            allow_writing_files=False, thread_count=1
        )))
    if HAS_XGB:
        bank.append(("xgb", xgb.XGBClassifier(
            n_estimators=120, learning_rate=0.025, max_depth=2, min_child_weight=2.0,
            subsample=0.85, colsample_bytree=0.80, reg_alpha=0.1, reg_lambda=3.0,
            objective="binary:logistic", eval_metric="logloss", random_state=seed,
            n_jobs=1, verbosity=0
        )))
    return bank


def regressor_bank(seed):
    bank = [
        ("ridge1", Pipeline([("sc", RobustScaler()), ("rg", Ridge(alpha=1.0))])),
        ("ridge5", Pipeline([("sc", StandardScaler()), ("rg", Ridge(alpha=5.0))])),
        ("br", Pipeline([("sc", StandardScaler()), ("rg", BayesianRidge())])),
        ("rf_r", RandomForestRegressor(n_estimators=120, max_depth=3, min_samples_leaf=3, max_features=0.65, random_state=seed, n_jobs=1)),
        ("et_r", ExtraTreesRegressor(n_estimators=180, max_depth=4, min_samples_leaf=2, max_features=0.75, random_state=seed, n_jobs=1)),
        ("gb_r", GradientBoostingRegressor(n_estimators=110, learning_rate=0.03, max_depth=2, subsample=0.85, random_state=seed)),
    ]
    if HAS_LGB:
        bank.append(("lgb_r", lgb.LGBMRegressor(
            n_estimators=150, learning_rate=0.025, num_leaves=5, max_depth=3,
            min_child_samples=5, subsample=0.85, colsample_bytree=0.80,
            reg_alpha=0.15, reg_lambda=2.0, random_state=seed, n_jobs=1, verbose=-1
        )))
    if HAS_CAT:
        bank.append(("cat_r", CatBoostRegressor(
            iterations=140, learning_rate=0.025, depth=3, l2_leaf_reg=8.0,
            loss_function="RMSE", verbose=False, random_seed=seed,
            allow_writing_files=False, thread_count=1
        )))
    if HAS_XGB:
        bank.append(("xgb_r", xgb.XGBRegressor(
            n_estimators=120, learning_rate=0.025, max_depth=2, min_child_weight=2.0,
            subsample=0.85, colsample_bytree=0.80, reg_alpha=0.1, reg_lambda=3.0,
            objective="reg:squarederror", random_state=seed, n_jobs=1, verbosity=0
        )))
    return bank


def safe_fit_predict_proba(model, X_tr, y_tr, X_va):
    y_tr = np.asarray(y_tr, dtype=int)
    if len(np.unique(y_tr)) < 2:
        return np.full(len(X_va), float(np.mean(y_tr)))
    try:
        model.fit(X_tr, y_tr)
        p = model.predict_proba(X_va)
        if isinstance(p, list):
            p = np.asarray(p)
        if p.ndim == 2 and p.shape[1] > 1:
            return np.asarray(p[:, 1], dtype=float)
        return np.asarray(p, dtype=float).reshape(-1)
    except Exception:
        return np.full(len(X_va), float(np.mean(y_tr)))


def safe_fit_predict_reg(model, X_tr, y_tr, X_va):
    try:
        model.fit(X_tr, y_tr)
        return np.asarray(model.predict(X_va), dtype=float).reshape(-1)
    except Exception:
        return np.full(len(X_va), float(np.mean(y_tr)))


def smooth_mean(total, count, prior, alpha=4.0):
    return (total + alpha * prior) / (count + alpha)


def grouped_te_candidate(y_near, horizon):
    prior = float(np.mean(y_near))
    train_raw = X.loc[NEAR, ["low_temporal_resolution_0_5h", "event_start_month", "event_start_hour", "hour_bucket", "log_area"]].copy().reset_index(drop=True)
    test_raw = X_test.loc[NEAR_TEST, ["low_temporal_resolution_0_5h", "event_start_month", "event_start_hour", "hour_bucket", "log_area"]].copy().reset_index(drop=True)
    train_raw["area_bin"] = pd.qcut(train_raw["log_area"].rank(method="first"), q=min(5, len(train_raw)), labels=False, duplicates="drop").astype(float)
    if len(test_raw):
        qs = np.quantile(train_raw["log_area"], [0.2, 0.4, 0.6, 0.8])
        test_raw["area_bin"] = np.searchsorted(qs, test_raw["log_area"].to_numpy(), side="right").astype(float)
    else:
        test_raw["area_bin"] = []

    key_sets = [
        ["low_temporal_resolution_0_5h"],
        ["low_temporal_resolution_0_5h", "event_start_month"],
        ["low_temporal_resolution_0_5h", "hour_bucket"],
        ["low_temporal_resolution_0_5h", "area_bin"],
        ["low_temporal_resolution_0_5h", "event_start_month", "hour_bucket"],
    ]
    key_weights = np.array([0.34, 0.22, 0.22, 0.12, 0.10], dtype=float)
    key_weights = key_weights / key_weights.sum()

    oof_near = np.zeros(len(NEAR_IDX), dtype=float)
    test_near = np.zeros(len(TEST_NEAR_IDX), dtype=float)

    for keys, kw in zip(key_sets, key_weights):
        # Leave-one-out train TE
        grp = train_raw.assign(y=y_near).groupby(keys)["y"].agg(["sum", "count"]).reset_index()
        merged = train_raw.merge(grp, on=keys, how="left")
        val = smooth_mean(merged["sum"].to_numpy() - y_near, merged["count"].to_numpy() - 1.0, prior, alpha=4.0)
        oof_near += kw * val

        if len(test_raw):
            full = train_raw.assign(y=y_near).groupby(keys)["y"].agg(["sum", "count"]).reset_index()
            tm = test_raw.merge(full, on=keys, how="left")
            tsum = tm["sum"].fillna(0.0).to_numpy()
            tcnt = tm["count"].fillna(0.0).to_numpy()
            test_near += kw * smooth_mean(tsum, tcnt, prior, alpha=4.0)

    oof = np.zeros(len(train), dtype=float)
    pred = np.zeros(len(test), dtype=float)
    oof[NEAR_IDX] = oof_near
    pred[TEST_NEAR_IDX] = test_near
    return oof, pred


def low_empirical_candidate(y_near):
    prior = float(np.mean(y_near))
    low_train = train.loc[NEAR, "low_temporal_resolution_0_5h"].to_numpy(dtype=int)
    low_test = test.loc[NEAR_TEST, "low_temporal_resolution_0_5h"].to_numpy(dtype=int)
    oof_near = np.zeros(len(NEAR_IDX), dtype=float)
    test_near = np.zeros(len(TEST_NEAR_IDX), dtype=float)
    for i in range(len(NEAR_IDX)):
        lv = low_train[i]
        m = (low_train == lv)
        oof_near[i] = smooth_mean(float(np.sum(y_near[m]) - y_near[i]), float(np.sum(m) - 1), prior, alpha=3.0)
    for i, lv in enumerate(low_test):
        m = (low_train == lv)
        test_near[i] = smooth_mean(float(np.sum(y_near[m])), float(np.sum(m)), prior, alpha=3.0)
    oof = np.zeros(len(train), dtype=float)
    pred = np.zeros(len(test), dtype=float)
    oof[NEAR_IDX] = oof_near
    pred[TEST_NEAR_IDX] = test_near
    return oof, pred


def build_horizon_candidates(horizon):
    y_full = ((EVENT == 1) & (TIME <= horizon)).astype(int)
    y_near = y_full[NEAR_IDX].astype(int)
    candidates_oof = []
    candidates_test = []
    names = []

    oof, pred = low_empirical_candidate(y_near)
    candidates_oof.append(oof); candidates_test.append(pred); names.append("low_emp")
    oof, pred = grouped_te_candidate(y_near, horizon)
    candidates_oof.append(oof); candidates_test.append(pred); names.append("group_te")

    min_class = int(np.min(np.bincount(y_near))) if len(np.unique(y_near)) == 2 else 0
    n_splits = min(5, min_class)
    if n_splits >= 2 and len(TEST_NEAR_IDX) > 0:
        sums_oof = {}
        counts_oof = {}
        sums_test = {}
        counts_test = {}
        for seed in SEEDS:
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            for tr_rel, va_rel in skf.split(X.loc[NEAR_IDX, FEATURES], y_near):
                tr_idx = NEAR_IDX[tr_rel]
                va_idx = NEAR_IDX[va_rel]
                for name, model in classifier_bank(seed):
                    if name not in sums_oof:
                        sums_oof[name] = np.zeros(len(train), dtype=float)
                        counts_oof[name] = np.zeros(len(train), dtype=float)
                    p = safe_fit_predict_proba(clone(model), X.loc[tr_idx, FEATURES], y_full[tr_idx], X.loc[va_idx, FEATURES])
                    sums_oof[name][va_idx] += p
                    counts_oof[name][va_idx] += 1.0
            # Full train-to-test candidate for this seed
            for name, model in classifier_bank(seed):
                if name not in sums_test:
                    sums_test[name] = np.zeros(len(test), dtype=float)
                    counts_test[name] = np.zeros(len(test), dtype=float)
                p_test = safe_fit_predict_proba(clone(model), X.loc[NEAR_IDX, FEATURES], y_full[NEAR_IDX], X_test.loc[TEST_NEAR_IDX, FEATURES])
                sums_test[name][TEST_NEAR_IDX] += p_test
                counts_test[name][TEST_NEAR_IDX] += 1.0

        for name in sorted(sums_oof.keys()):
            oof = np.zeros(len(train), dtype=float)
            pred = np.zeros(len(test), dtype=float)
            m = counts_oof[name] > 0
            oof[m] = sums_oof[name][m] / counts_oof[name][m]
            miss = NEAR & (~m)
            if np.any(miss):
                oof[miss] = float(np.mean(y_near))
            mt = counts_test[name] > 0
            pred[mt] = sums_test[name][mt] / counts_test[name][mt]
            candidates_oof.append(oof)
            candidates_test.append(pred)
            names.append(name)
    else:
        # Degenerate fallback: no enough class variety
        oof = np.zeros(len(train), dtype=float)
        pred = np.zeros(len(test), dtype=float)
        oof[NEAR_IDX] = float(np.mean(y_near))
        pred[TEST_NEAR_IDX] = float(np.mean(y_near))
        candidates_oof.append(oof); candidates_test.append(pred); names.append("near_mean")

    return names, np.column_stack(candidates_oof), np.column_stack(candidates_test), y_full


def optimize_blend(M, y_full, horizon):
    valid = ~((EVENT == 0) & (TIME < horizon))
    y = y_full.astype(float)
    k = M.shape[1]
    prior = np.ones(k, dtype=float) / k

    def softmax(z):
        z = np.asarray(z, dtype=float)
        z = z - np.max(z)
        e = np.exp(z)
        return e / np.sum(e)

    def obj(z):
        w = softmax(z)
        p = np.clip(M @ w, 0.0, 1.0)
        return float(np.mean((p[valid] - y[valid]) ** 2) + 0.0015 * np.sum((w - prior) ** 2))

    best = None
    for scale in [0.0, 0.15, -0.15]:
        init = np.zeros(k, dtype=float) + scale
        res = minimize(obj, init, method="BFGS", options={"maxiter": 600, "gtol": 1e-7})
        if best is None or res.fun < best.fun:
            best = res
    return softmax(best.x)

P0_OOF = np.zeros((len(train), 4), dtype=float)
P0_TEST = np.zeros((len(test), 4), dtype=float)
P0_OOF[:, 3] = 1.0
P0_TEST[:, 3] = 1.0
blend_log = {}

for col, H in enumerate(HORIZONS):
    names, M_oof, M_test, y_full = build_horizon_candidates(H)
    w = optimize_blend(M_oof, y_full, H)
    P0_OOF[:, col] = np.clip(M_oof @ w, 0.0, 1.0)
    P0_TEST[:, col] = np.clip(M_test @ w, 0.0, 1.0)
    blend_log[H] = [(n, float(wi)) for n, wi in zip(names, w)]
    print("H", H, "Brier", compute_brier(TIME, EVENT, P0_OOF[:, col], H), "top weights", [(n, round(float(wi), 3)) for n, wi in zip(names, w) if wi > 0.03])

P0_OOF = enforce_monotone(P0_OOF)
P0_TEST = enforce_monotone(P0_TEST)
P0_OOF[:, 3] = 1.0
P0_TEST[:, 3] = 1.0
print("Base OOF", hybrid_score(TIME, EVENT, P0_OOF))


def build_time_rank_signal():
    ylog = np.log1p(TIME)
    sums_oof = {}
    counts_oof = {}
    sums_test = {}
    counts_test = {}
    for seed in SEEDS:
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_rel, va_rel in kf.split(NEAR_IDX):
            tr_idx = NEAR_IDX[tr_rel]
            va_idx = NEAR_IDX[va_rel]
            for name, model in regressor_bank(seed):
                if name not in sums_oof:
                    sums_oof[name] = np.zeros(len(train), dtype=float)
                    counts_oof[name] = np.zeros(len(train), dtype=float)
                p = safe_fit_predict_reg(clone(model), X.loc[tr_idx, FEATURES], ylog[tr_idx], X.loc[va_idx, FEATURES])
                sums_oof[name][va_idx] += p
                counts_oof[name][va_idx] += 1.0
        for name, model in regressor_bank(seed):
            if name not in sums_test:
                sums_test[name] = np.zeros(len(test), dtype=float)
                counts_test[name] = np.zeros(len(test), dtype=float)
            p_test = safe_fit_predict_reg(clone(model), X.loc[NEAR_IDX, FEATURES], ylog[NEAR_IDX], X_test.loc[TEST_NEAR_IDX, FEATURES])
            sums_test[name][TEST_NEAR_IDX] += p_test
            counts_test[name][TEST_NEAR_IDX] += 1.0

    pred_oofs = []
    pred_tests = []
    weights = []
    rank_log = []
    large_time = float(np.nanmedian(ylog[NEAR_IDX]) + 8.0)
    for name in sorted(sums_oof.keys()):
        oof = np.full(len(train), large_time, dtype=float)
        pred = np.full(len(test), large_time, dtype=float)
        m = counts_oof[name] > 0
        oof[m] = sums_oof[name][m] / counts_oof[name][m]
        mt = counts_test[name] > 0
        pred[mt] = sums_test[name][mt] / counts_test[name][mt]
        cidx = compute_c_index(TIME, EVENT, -oof)
        w = max(cidx - 0.5, 1e-4) ** 2
        pred_oofs.append(oof)
        pred_tests.append(pred)
        weights.append(w)
        rank_log.append((name, float(cidx), float(w)))

    if len(weights) == 0 or np.sum(weights) <= 0:
        return np.zeros(len(train), dtype=float), np.zeros(len(test), dtype=float), []
    weights = np.asarray(weights, dtype=float)
    weights = weights / np.sum(weights)
    score_oof = np.tensordot(weights, np.vstack(pred_oofs), axes=(0, 0))
    score_test = np.tensordot(weights, np.vstack(pred_tests), axes=(0, 0))
    rank_log = [(n, c, float(w)) for (n, c, _), w in zip(rank_log, weights)]
    return score_oof, score_test, rank_log

time_score_oof, time_score_test, rank_log = build_time_rank_signal()
print("Time-rank top", sorted([(n, round(c, 4), round(w, 3)) for n, c, w in rank_log], key=lambda z: -z[2])[:8])

base_risk_oof = 0.3 * P0_OOF[:, 1] + 0.4 * P0_OOF[:, 2]
base_risk_test = 0.3 * P0_TEST[:, 1] + 0.4 * P0_TEST[:, 2]

def standardize_from_train(train_values, test_values, mask):
    mu = float(np.mean(train_values[mask]))
    sd = float(np.std(train_values[mask]) + 1e-8)
    return (train_values - mu) / sd, (test_values - mu) / sd

z_base_oof, z_base_test = standardize_from_train(base_risk_oof, base_risk_test, NEAR)
z_time_oof, z_time_test = standardize_from_train(-time_score_oof, -time_score_test, NEAR)
RISK_Z_OOF = 0.55 * z_base_oof + 0.45 * z_time_oof
RISK_Z_TEST = 0.55 * z_base_test + 0.45 * z_time_test


def apply_rank_transform(Pbase, near_mask, risk_z, params):
    P = np.zeros_like(Pbase, dtype=float)
    P[:, 3] = 1.0
    a = params[:3]
    b = params[3:6]
    g = params[6:9]
    for j in range(3):
        z = a[j] * logit(Pbase[:, j]) + b[j] + g[j] * risk_z
        pj = sigmoid(z)
        P[:, j] = np.where(near_mask, pj, 0.0)
    P = enforce_monotone(P)
    P[:, 3] = 1.0
    return P

INIT_PARAMS = np.array([1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=float)


def transform_objective(params):
    P = apply_rank_transform(P0_OOF, NEAR, RISK_Z_OOF, params)
    score = hybrid_score(TIME, EVENT, P)[0]
    reg = 0.0012 * float(np.sum((params - INIT_PARAMS) ** 2))
    return -score + reg

best = None
rng = np.random.default_rng(SEED)
starts = [INIT_PARAMS]
for _ in range(8):
    starts.append(INIT_PARAMS + rng.normal(0.0, 0.18, size=9))

for i, start in enumerate(starts):
    res = minimize(transform_objective, start, method="Nelder-Mead", options={"maxiter": 2200, "xatol": 1e-7, "fatol": 1e-8})
    if best is None or transform_objective(res.x) < transform_objective(best.x):
        best = res
        print("Rank transform", i, hybrid_score(TIME, EVENT, apply_rank_transform(P0_OOF, NEAR, RISK_Z_OOF, best.x)))

P_OOF = apply_rank_transform(P0_OOF, NEAR, RISK_Z_OOF, best.x)
P_TEST = apply_rank_transform(P0_TEST, NEAR_TEST, RISK_Z_TEST, best.x)
P_TEST[:, 3] = 1.0
P_TEST = enforce_monotone(P_TEST)
P_TEST[:, 3] = 1.0

print("Final OOF", hybrid_score(TIME, EVENT, P_OOF))
print("Transform params", np.round(best.x, 6).tolist())

submission = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h": P_TEST[:, 0],
    "prob_24h": P_TEST[:, 1],
    "prob_48h": P_TEST[:, 2],
    "prob_72h": P_TEST[:, 3],
})
submission = sample[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

# Strict contest-schema validation
assert list(submission.columns) == REQUIRED
assert len(submission) == len(sample)
assert submission["event_id"].equals(sample["event_id"])
assert submission["event_id"].is_unique
vals = submission[REQUIRED[1:]].to_numpy(dtype=float)
assert np.isfinite(vals).all()
assert ((vals >= 0.0) & (vals <= 1.0)).all()
assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
submission.to_csv("submission.csv", index=False)
print("Saved", OUTPUT_PATH)
print(submission.head(10).to_string(index=False))


DATA_DIR /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
train/test (221, 37) (95, 35)
train event sum 69 train <5km 69 test <5km 28
H 12 Brier 0.048251627808080164 top weights [('low_emp', 0.353), ('group_te', 0.212), ('ada', 0.139), ('et', 0.296)]
H 24 Brier 0.01867255092751711 top weights [('cat', 0.108), ('et', 0.155), ('gb', 0.356), ('lgb', 0.381)]
H 48 Brier 0.01015656648225797 top weights [('group_te', 0.151), ('et', 0.585), ('lgb', 0.102), ('rf', 0.118), ('xgb', 0.038)]
Base OOF (0.9745306860379053, 0.936816826763829, 0.009306231416062117, 0.01887672767137927, 0.009108032786620838, 0.0)
Time-rank top [('gb_r', 0.9408, 0.116), ('lgb_r', 0.942, 0.116), ('rf_r', 0.941, 0.116), ('xgb_r', 0.9414, 0.116), ('br', 0.9391, 0.115), ('cat_r', 0.9393, 0.115), ('et_r', 0.9382, 0.114), ('ridge5', 0.9338, 0.112)]
Rank transform 0 (0.9747208680220723, 0.9373136800264988, 0.009247479979824743, 0.018736663359612375, 0.009066202429852578, 0.0)
Rank transform 1 (0.9781849496909655, 0.94857